# Relationformer su Kaggle: Run All con due GPU

Imposta **GPU T4 ×2** (o due GPU CUDA), collega il dataset patched PID2Graph,
abilita **Internet**, quindi esegui **Run All**. Il dataset deve essere già estratto in PNG/GraphML.

Impostazioni iniziali: **batch 4 per GPU (8 complessivo), AMP, worker loader 0,
preprocessing 2 processi, blocchi archi 2048, budget training massimo 9 ore**.
Il dataset viene cercato automaticamente. Modifica `DATASET_ROOT_OVERRIDE` solo se ci sono più copie valide.

Il notebook include una copia dei sorgenti della revisione locale: non richiede clone, branch o push.
Le librerie e i pesi ImageNet possono essere scaricati via Internet; PyTorch/CUDA di Kaggle vengono mantenuti.
La directory dei sorgenti viene creata per ogni esecuzione; la cache del dataset viene riutilizzata.
Questo rimane un esperimento sui dati disponibili, non una replica esatta del protocollo del paper.

Questa versione corregge l’errore CPU `rebuild_storage_fd: unable to mmap`.
Se una precedente esecuzione è rimasta bloccata, termina la sessione prima di ripartire.


## 1. Opzioni della sessione e ripresa


In [1]:
import os
import sys
import time
from pathlib import Path

NOTEBOOK_STARTED = time.monotonic()
DATASET_ROOT_OVERRIDE = None
RESUME_CHECKPOINT = None  # Esempio: "/kaggle/input/mio-checkpoint/checkpoint_epoch=12.pt"
BATCH_PER_GPU = 4
TRAIN_MAX_HOURS = 9.0
SESSION_HOURS_REMAINING = 11.0  # Stima all'avvio: circa 1 ora già usata su 12.
SESSION_MARGIN_HOURS = 1.0
SEED = 10

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["RELATIONFORMER_PREPROCESS_WORKERS"] = "2"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["MPLBACKEND"] = "Agg"
os.environ["RELATIONFORMER_CACHE_DIR"] = "/kaggle/working/relationformer-cache"
Path(os.environ["RELATIONFORMER_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

if RESUME_CHECKPOINT is not None:
    RESUME_CHECKPOINT = str(Path(RESUME_CHECKPOINT).resolve())
    assert Path(RESUME_CHECKPOINT).is_file(), f"Checkpoint non trovato: {RESUME_CHECKPOINT}"
assert BATCH_PER_GPU >= 1
assert TRAIN_MAX_HOURS > 0
assert SESSION_HOURS_REMAINING > SESSION_MARGIN_HOURS >= 0


## 2. Sorgenti inclusi nel notebook

La cella contiene un archivio compresso dei file Python, della config e dei requirements.
Verifica SHA-256 ed estrae solo percorsi relativi nella nuova directory di lavoro.
Non cancella repository o risultati di esecuzioni precedenti.


In [2]:
import base64
import hashlib
import io
import tempfile
import zipfile

SOURCE_BASE_COMMIT = 'ab179577e24df462147f2999119c20c7e6e1f7ac'
SOURCE_SNAPSHOT_SHA256 = '63a734e3ef7c8029eb3f07df86d85c0cf303729c1aa1fb5842d023bbfd00f8fa'
SOURCE_ARCHIVE_B64 = '...'

snapshot = base64.b64decode(SOURCE_ARCHIVE_B64)
assert hashlib.sha256(snapshot).hexdigest() == SOURCE_SNAPSHOT_SHA256
WORKDIR = Path(tempfile.mkdtemp(prefix="relationformer-", dir="/kaggle/working"))
with zipfile.ZipFile(io.BytesIO(snapshot)) as archive:
    for member in archive.infolist():
        relative = Path(member.filename)
        assert not relative.is_absolute() and ".." not in relative.parts
    archive.extractall(WORKDIR)
os.chdir(WORKDIR)
print("Sorgenti:", WORKDIR)
print("Snapshot SHA-256:", SOURCE_SNAPSHOT_SHA256)


Sorgenti: /kaggle/working/relationformer-u914xqw4
Snapshot SHA-256: 63a734e3ef7c8029eb3f07df86d85c0cf303729c1aa1fb5842d023bbfd00f8fa


## 3. Verifica delle due GPU e dipendenze


In [3]:
import subprocess
from importlib.metadata import version
import torch

assert torch.cuda.is_available(), "Abilita l'acceleratore GPU nelle impostazioni Kaggle."
assert torch.cuda.device_count() == 2, "Questo profilo richiede due GPU: seleziona T4 x2."
gpu_ids = ["0", "1"]
print("Python:", sys.version.split()[0], "Torch:", torch.__version__, "CUDA:", torch.version.cuda)
for index in range(2):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name}, {props.total_memory / 2**30:.1f} GiB")

# Impedisce che una dipendenza aggiorni silenziosamente Torch/torchvision di Kaggle.
constraints = WORKDIR / "kaggle-torch-constraints.txt"
constraints.write_text(f"torch=={version('torch')}\ntorchvision=={version('torchvision')}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(WORKDIR / "requirements-kaggle.txt"),
                "-c", str(constraints)], check=True)
import yaml
print("Dipendenze installate; Torch/torchvision mantenuti.")


Python: 3.12.13 Torch: 2.10.0+cu128 CUDA: 12.8
GPU 0: Tesla T4, 14.6 GiB
GPU 1: Tesla T4, 14.6 GiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.0 MB/s eta 0:00:00
Dipendenze installate; Torch/torchvision mantenuti.


## 4. Dataset e configurazione

Cerca la root contenente `Dataset PID`, `PID2Graph Synthetic`, `PID2Graph OPEN100`.
OPEN100 rimane nel test. Il training usa tutti i campioni ammessi dai filtri del loader;
non viene applicato un sottocampionamento aggiuntivo.


In [4]:
REQUIRED_SOURCES = {"Dataset PID", "PID2Graph Synthetic", "PID2Graph OPEN100"}
candidates = ([Path(DATASET_ROOT_OVERRIDE)] if DATASET_ROOT_OVERRIDE else sorted({
    marker.parent for marker in Path("/kaggle/input").rglob("Dataset PID") if marker.is_dir()
}))
valid_roots = [path for path in candidates if path.is_dir()
               and all((path / name).is_dir() for name in REQUIRED_SOURCES)]
assert len(valid_roots) == 1, (
    f"Root valide: {[str(path) for path in valid_roots]}. "
    "Collega il dataset patched o imposta DATASET_ROOT_OVERRIDE nella prima cella."
)
dataset_root = valid_roots[0].resolve()
os.environ["PID2GRAPH_DATA_PATH"] = str(dataset_root)
source_stats = {}
for name in sorted(REQUIRED_SOURCES):
    graphs = list((dataset_root / name).rglob("*.graphml"))
    paired = sum(path.with_suffix(".png").is_file() for path in graphs)
    assert graphs and paired == len(graphs), f"PNG/GraphML mancanti in {name}"
    source_stats[name] = {"graphml": len(graphs), "paired": paired}
print("Dataset:", dataset_root)
print("Campioni per sorgente:", source_stats)

with (WORKDIR / "configs/road_2D.yaml").open() as f:
    config = yaml.safe_load(f)
config["DATA"].update(BATCH_SIZE=BATCH_PER_GPU, NUM_WORKERS=0, SEED=SEED,
                      DATA_PATH=str(dataset_root), CACHE_DIR=os.environ["RELATIONFORMER_CACHE_DIR"])
config["TRAIN"].update(AMP=True, MAX_HOURS=TRAIN_MAX_HOURS, LOG_INTERVAL=10,
                       SAVE_PATH=str(WORKDIR / "trained_weights"))
config["INFERENCE"]["EDGE_CHUNK_SIZE"] = 2048
CONFIG_PATH = WORKDIR / "configs/kaggle_session.yaml"
with CONFIG_PATH.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
print("Batch per GPU:", BATCH_PER_GPU, "Batch globale:", BATCH_PER_GPU * 2)
print("Config:", CONFIG_PATH)


Dataset: /kaggle/input/datasets/simonesgalla/patched
Campioni per sorgente: {'Dataset PID': {'graphml': 19462, 'paired': 19462}, 'PID2Graph OPEN100': {'graphml': 1629, 'paired': 1629}, 'PID2Graph Synthetic': {'graphml': 14708, 'paired': 14708}}
Batch per GPU: 4 Batch globale: 8
Config: /kaggle/working/relationformer-u914xqw4/configs/kaggle_session.yaml


## 5. Preparazione cache e controllo del modello

Il preprocessing usa CPU e disco: GPU poco utilizzate in questa fase sono normali.
Lo snapshot degli split viene salvato insieme ai risultati.
Il controllo del modello verifica costruzione e caricamento, non il picco di memoria del backward.

Correzione memoria CPU: i worker trasferiscono i grafi come NumPy, senza creare
una memoria condivisa Torch per ogni tensore. Con
`RELATIONFORMER_PREPROCESS_WORKERS=1` il preprocessing è interamente seriale.
Le cache completate restano riutilizzabili; i file di cache incompleti vengono ricostruiti.


In [5]:
preprocess_code = r"""
import json
import sys
from pathlib import Path
from train import load_config
from dataset_road_network import build_road_network_data

cfg = load_config(sys.argv[1], verbose=False)
train, val = build_road_network_data(cfg, mode='split')
assert len(train) and len(val)
groups = lambda ds: {tuple(sample.split('/')[:2]) for sample in ds.ids}
assert not groups(train) & groups(val)
run = Path(cfg.TRAIN.SAVE_PATH) / 'runs' / f'{cfg.log.exp_name}_{cfg.DATA.SEED}'
run.mkdir(parents=True, exist_ok=True)
(run / 'split_manifest.json').write_text(json.dumps({'train': train.ids, 'validation': val.ids}))
print('Train:', len(train), 'Validation:', len(val), flush=True)
"""
subprocess.run([sys.executable, "-u", "-c", preprocess_code, str(CONFIG_PATH)], cwd=WORKDIR, check=True)

model_preflight_code = r"""
import sys
import torch
from train import load_config
from models import build_model
from evaluator import build_evaluator
from trainer import build_trainer
cfg = load_config(sys.argv[1], verbose=False)
if len(sys.argv) > 2:
    cfg.MODEL.ENCODER.PRETRAINED = False
model = build_model(cfg).cuda()
if len(sys.argv) > 2:
    checkpoint = torch.load(sys.argv[2], map_location='cpu', weights_only=True)
    model.load_state_dict(checkpoint['net'])
print('Parametri addestrabili:', sum(p.numel() for p in model.parameters() if p.requires_grad), flush=True)
"""
preflight_command = [sys.executable, "-u", "-c", model_preflight_code, str(CONFIG_PATH)]
if RESUME_CHECKPOINT is not None:
    preflight_command.append(RESUME_CHECKPOINT)
subprocess.run(preflight_command, cwd=WORKDIR, check=True)


Indexing P&ID samples under /kaggle/input/datasets/simonesgalla/patched (first run, this can take a few minutes)...
Found 35799 GraphML files, filtering...
  5000/35799 graphs processed (48s)
  10000/35799 graphs processed (75s)
  15000/35799 graphs processed (99s)
  20000/35799 graphs processed (122s)
  25000/35799 graphs processed (141s)
  30000/35799 graphs processed (161s)
  35000/35799 graphs processed (181s)
Indexing done in 184s (94 unusable graphs skipped): test=1585, train=32616, valid=1504
Preprocessing 32616 P&ID samples into /kaggle/working/relationformer-cache/pid_95ce4eb55ff34a6e_train_* (first run)...
Preprocessing workers=2; graph transport=NumPy (no shared tensor IPC)
  5000/32616 samples preprocessed (107s)
  10000/32616 samples preprocessed (213s)
  15000/32616 samples preprocessed (322s)
  20000/32616 samples preprocessed (447s)
  25000/32616 samples preprocessed (607s)
  30000/32616 samples preprocessed (762s)
Preprocessing done in 852s
Loaded 32616 P&ID samples fo

/kaggle/working/relationformer-u914xqw4/models/ops/modules/ms_deform_attn.py:90: SyntaxWarning: invalid escape sequence '\s'
  :param input_flatten               (N, \sum_{l=0}^{L-1} H_l \cdot W_l, C)
/kaggle/working/relationformer-u914xqw4/metric_smd.py:510: SyntaxWarning: invalid escape sequence '\e'
  "$M_{ij} = (-c_{ij} + u_i + v_j) / \epsilon$"


Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:00<00:00, 213MB/s]


Parametri addestrabili: 55403540


CompletedProcess(args=['/usr/bin/python3', '-u', '-c', "\nimport sys\nimport torch\nfrom train import load_config\nfrom models import build_model\nfrom evaluator import build_evaluator\nfrom trainer import build_trainer\ncfg = load_config(sys.argv[1], verbose=False)\nif len(sys.argv) > 2:\n    cfg.MODEL.ENCODER.PRETRAINED = False\nmodel = build_model(cfg).cuda()\nif len(sys.argv) > 2:\n    checkpoint = torch.load(sys.argv[2], map_location='cpu', weights_only=True)\n    model.load_state_dict(checkpoint['net'])\nprint('Parametri addestrabili:', sum(p.numel() for p in model.parameters() if p.requires_grad), flush=True)\n", '/kaggle/working/relationformer-u914xqw4/configs/kaggle_session.yaml'], returncode=0)

## 6. Training o ripresa: una sola esecuzione

Usa le GPU 0 e 1. `RESUME_CHECKPOINT` nella prima cella sceglie se iniziare o riprendere.
Il budget è al massimo 9 ore e viene ridotto se la preparazione ha consumato il margine stimato.
Il limite effettivo Kaggle rimane esterno: il trainer controlla il tempo a fine epoca,
quindi non garantisce il salvataggio prima di una terminazione forzata.

Se compare CUDA OOM, riduci `BATCH_PER_GPU` a 2 e riparti dall'ultimo checkpoint disponibile.


In [6]:
import json
import shutil

elapsed_hours = (time.monotonic() - NOTEBOOK_STARTED) / 3600
remaining_budget = SESSION_HOURS_REMAINING - elapsed_hours - SESSION_MARGIN_HOURS
assert remaining_budget > 0, "Tempo stimato insufficiente: salva gli output e usa una nuova sessione."
config["TRAIN"]["MAX_HOURS"] = round(min(TRAIN_MAX_HOURS, remaining_budget), 4)
assert config["TRAIN"]["MAX_HOURS"] > 0
with CONFIG_PATH.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

run_dir = Path(config["TRAIN"]["SAVE_PATH"]) / "runs" / f"{config['log']['exp_name']}_{SEED}"
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(CONFIG_PATH, run_dir / "kaggle_session.yaml")
(run_dir / "environment.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))
manifest = {
    "source_base_commit": SOURCE_BASE_COMMIT,
    "source_snapshot_sha256": SOURCE_SNAPSHOT_SHA256,
    "source_includes_local_changes": True,
    "dataset_root": str(dataset_root), "dataset_stats": source_stats,
    "torch_version": torch.__version__, "cuda_version": torch.version.cuda,
    "gpus": [torch.cuda.get_device_name(i) for i in range(2)],
    "config": config, "resume_checkpoint": RESUME_CHECKPOINT,
}
(run_dir / "kaggle_manifest.json").write_text(json.dumps(manifest, indent=2))
(run_dir / "source_snapshot.zip").write_bytes(snapshot)

training_command = [sys.executable, "-u", "train.py", "--config", str(CONFIG_PATH),
                    "--cuda_visible_device", *gpu_ids]
if RESUME_CHECKPOINT is not None:
    training_command.extend(["--resume", RESUME_CHECKPOINT])
print("Budget training:", config["TRAIN"]["MAX_HOURS"], "ore", flush=True)
print("Avvio:", " ".join(training_command), flush=True)
print("Output:", run_dir, flush=True)
try:
    subprocess.run(training_command, cwd=WORKDIR, check=True)
finally:
    # Utile anche se il sottoprocesso restituisce un errore; una chiusura forzata
    # dell'intera sessione Kaggle può impedire l'esecuzione di questo blocco.
    archive_base = Path("/kaggle/working") / f"{WORKDIR.name}_artifacts"
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=run_dir)
    print("Archivio output:", archive_path, flush=True)


Budget training: 9.0 ore
Avvio: /usr/bin/python3 -u train.py --config /kaggle/working/relationformer-u914xqw4/configs/kaggle_session.yaml --cuda_visible_device 0 1
Output: /kaggle/working/relationformer-u914xqw4/trained_weights/runs/pid2graph_kaggle_10

*** Config file
/kaggle/working/relationformer-u914xqw4/configs/kaggle_session.yaml
Relationformer multi-class training on patched PID2Graph for Kaggle
Loaded P&ID sample index from /kaggle/working/relationformer-cache/pid_index_3be02b7928dd5c74.pickle
Loaded preprocessed P&ID samples from /kaggle/working/relationformer-cache/pid_95ce4eb55ff34a6e_train_*
Loaded 32616 P&ID samples for train.
Loaded P&ID sample index from /kaggle/working/relationformer-cache/pid_index_3be02b7928dd5c74.pickle
Loaded preprocessed P&ID samples from /kaggle/working/relationformer-cache/pid_731bc89fa9c1e117_valid_*
Loaded 1504 P&ID samples for valid.


2026-09-10 10:14:03,394 ignite.distributed.launcher.Parallel INFO: Initialized distributed launcher with backend: 'nccl'
2026-09-10 10:14:03,394 ignite.distributed.launcher.Parallel INFO: - Parameters to spawn processes: 
	nproc_per_node: 2
	nnodes: 1
	node_rank: 0
2026-09-10 10:14:03,394 ignite.distributed.launcher.Parallel INFO: Spawn function '<function training at 0x7ee813f4c400>' in 2 processes


[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
world_size=2 amp=True device=cuda:0
Loaded P&ID sample index from /kaggle/working/relationformer-cache/pid_index_3be02b7928dd5c74.pickle
Loaded P&ID sample index from /kaggle/working/relationformer-cache/pid_index_3be02b7928dd5c74.pickle
Loaded preprocessed P&ID samples from /kaggle/working/relationformer-cache/pid_95ce4eb55ff34a6e_train_*
Loaded 32616 P&ID samples for train.
Loaded P&ID sample index from /kaggle/working/relationformer-cache/pid_index_3be02b7928dd5c74.pickle
Loaded preprocessed P&ID samples from /kaggle/working/relationformer-cache/pid_95ce4eb55ff34a6e_train_*
Loaded 32616 P&ID samples for train.
Loaded P&ID sample index from /kaggle/working/relationformer-cache/pid_index_3be02b7928dd5c74.pickle
Loaded preprocessed P&ID samples from /kaggle/working/relationformer-cache/pid_731bc89fa9c1e117_

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/kaggle/working/relationformer-u914xqw4/trainer.py:46: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  engine.state.log_sum[key] = engine.state.log_sum.get(key, 0.0) + floa

10:15:08 INFO relationformer.train: epoch 1/80 iter 10/4077 class=0.9539 nodes=0.4931 boxes=1.1652 edges=0.6581 cards=0.0862 total=8.2712 lr=1.00e-04 0.72 it/s eta 95 min
10:15:14 INFO relationformer.train: epoch 1/80 iter 20/4077 class=0.8553 nodes=0.4079 boxes=1.0643 edges=0.5740 cards=0.0312 total=7.2236 lr=1.00e-04 1.02 it/s eta 66 min
10:15:19 INFO relationformer.train: epoch 1/80 iter 30/4077 class=0.7252 nodes=0.2687 boxes=1.0065 edges=0.5664 cards=0.0206 total=6.2579 lr=1.00e-04 1.18 it/s eta 57 min
10:15:25 INFO relationformer.train: epoch 1/80 iter 40/4077 class=0.9350 nodes=0.1372 boxes=0.9984 edges=0.5322 cards=0.0388 total=6.0509 lr=1.00e-04 1.29 it/s eta 52 min
10:15:31 INFO relationformer.train: epoch 1/80 iter 50/4077 class=0.9177 nodes=0.0865 boxes=0.9985 edges=0.5575 cards=0.0187 total=5.8699 lr=1.00e-04 1.36 it/s eta 49 min
10:15:36 INFO relationformer.train: epoch 1/80 iter 60/4077 class=0.9387 nodes=0.0640 boxes=0.9989 edges=0.5415 cards=0.0619 total=5.8175 lr=1.00

Exception in thread Thread-1:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.12/dist-packages/tensorboardX/event_file_writer.py", line 199, in run
    data = self._queue.get(True, queue_wait_duration)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 117, in get
    res = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 212, in recv_bytes
    self._check_closed()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed
2026-09-10 18:42:16,684 ignite.distributed.launcher.Parallel INFO: End of run


Archivio output: /kaggle/working/relationformer-u914xqw4_artifacts.zip


## 7. Risultati

Scarica l'archivio `relationformer-..._artifacts.zip` dagli output di `/kaggle/working`.
Contiene checkpoint disponibili, log, configurazione effettiva, versioni delle librerie,
manifest degli split e copia esatta dei sorgenti eseguiti. La cache non è inclusa.
Salva la versione/output del notebook prima della scadenza della sessione.


In [7]:
checkpoints = sorted((run_dir / "models").glob("*.pt"), key=lambda p: p.stat().st_mtime)
for path in checkpoints:
    print(path.name, f"{path.stat().st_size / 2**20:.1f} MiB")
print("Log:", run_dir / "train.log")
print("Archivio:", archive_path)


checkpoint_key_metric=-0.0507.pt 621.1 MiB
checkpoint_key_metric=-0.0454.pt 621.1 MiB
checkpoint_key_metric=-0.0453.pt 621.1 MiB
checkpoint_key_metric=-0.0441.pt 621.1 MiB
checkpoint_key_metric=-0.0404.pt 621.1 MiB
checkpoint_epoch=12.pt 621.1 MiB
Log: /kaggle/working/relationformer-u914xqw4/trained_weights/runs/pid2graph_kaggle_10/train.log
Archivio: /kaggle/working/relationformer-u914xqw4_artifacts.zip
